# 1 · Meet the Beast

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=01-meet-the-beast.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/01-meet-the-beast.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** on
the site to switch story / how-to / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — your companion creature
:class: storytelling

*You are an adventurer, and every adventurer keeps a companion creature. You were
entrusted with **the Beast** — powerful, but not yet yours to command. Today the
masters simply **introduce** you: they show, on the Beast itself, what it can do.
They warm it from within, then grab a metal plate and stretch it to show how
robustly the Beast handles even large deformations. You only watch — the real
training starts next unit.*
:::

Almost every NGSolve session is the **same three steps**, no matter how hard the
problem: **(1)** conjure a **geometry and mesh**, **(2)** **solve a variational
problem** on it, **(3)** **visualize** the result. We will walk that loop **twice** —
once for a linear **Poisson** problem and once for a **nonlinear elasticity** problem —
so you see the shape of everything to come. Features fly past (boundary conditions,
solvers, nonlinear iterations); we name them and move on. Each gets its own unit later.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve import solvers
from ngsolve.webgui import Draw

## 1. Geometry & mesh — the Beast

The Beast is NGSolve's logo **sculpture**, forged in four boolean steps: take a **sphere**,
**hollow** it by subtracting a smaller inner sphere (a thick shell), then **bore** three
cylinders through it — what is left is the Beast.

```{image} data/beast-construction.jpg
:alt: Building the Beast — a sphere, hollowed into a shell, bored by three cylinders, the result
:width: 760px
:align: center
```

We build exactly that with **Netgen/OCC** constructive geometry (sphere and cylinder
primitives combined with boolean `-`), then let Netgen mesh it. Naming a few faces now lets
us attach boundary conditions later.

In [ ]:
def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)     # centre + shrink

beast = beast_sculpture()
mesh = Mesh(OCCGeometry(beast).GenerateMesh(maxh=0.7))
mesh.Curve(2)
print(f"the Beast: {mesh.nv} vertices, {mesh.ne} elements")
Draw(mesh)

## 2. A variational solve — fire-energy inside the Beast

Legend says the Beast stores the energy for its fire-breath deep in its body. We model
that as a **heat source** living inside the shell and ask for the steady temperature —
the **Poisson problem** $-\Delta u = f$ with $u=0$ on the surface. In NGSolve this is a
**weak form** on an `H1` space: find $u$ such that
$$ \underbrace{\int_\Omega \nabla u\cdot\nabla v}_{a(u,\,v)} \;=\; \underbrace{\int_\Omega f\,v}_{f(v)} \qquad\text{for all } v. $$
Recent NGSolve lets you write *exactly that* — a **bilinear form `== ` a linear form** —
and hand it to a single **`Solve`**. Read the two key lines as the maths above:

```{image} data/beast-heatsource.png
:alt: The fire-energy as a warm source glowing deep inside the translucent Beast
:width: 300px
:align: center
```
<p style="text-align:center"><sub>The heat source <code>f</code> — a glow buried in the
Beast's shell. The solve spreads it into a temperature field that vanishes on the surface.</sub></p>

In [ ]:
r = sqrt(x*x + y*y + z*z)
source = 40 * exp(-((r - 3.25) / 0.5)**2)            # a glow buried in the shell

fes = H1(mesh, order=2, dirichlet=".*")              # u = 0 on the whole surface
u, v = fes.TnT()                                     # trial & test functions
a = BilinearForm(grad(u) * grad(v) * dx)             # the bilinear form  a(u,v) = ∫ ∇u·∇v
f = LinearForm(source * v * dx)                      # the linear form    f(v)   = ∫ source·v
gfu = GridFunction(fes)                              # will hold the solution u

pre = Preconditioner(a, "local")                     # a cheap preconditioner (solvers: unit 6)
Solve(a * gfu == f, pre=pre)                         # one line: assemble, apply BCs, iterate
print(f"hottest point inside the Beast: {max(gfu.vec):.2f}")
Draw(gfu, mesh, "temperature")

### The same solve, unpacked — meet the detail objects

That one-liner is convenient, but the rest of this tutorial works directly with the
**objects it hides**, so let us redo the solve **by hand** and name them. `Solve`
essentially **assembles** the bilinear form into a sparse **matrix** `a.mat` and the
linear form into a **load vector** `f.vec`, then solves the linear system on the **free**
(non-Dirichlet) dofs — `fes.FreeDofs()`. Spelled out, with a direct factorisation:

In [ ]:
a.Assemble()                                         # BilinearForm  →  sparse matrix  a.mat
f.Assemble()                                         # LinearForm    →  load vector    f.vec
gfu_manual = GridFunction(fes)                       # its coefficients live in .vec
gfu_manual.vec.data = \
    a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec   # solve on the free dofs
print(f"by hand, the same hottest point: {max(gfu_manual.vec):.2f}")

Same answer, two routes. These pieces — **`BilinearForm`**, **`LinearForm`**,
**`GridFunction`**, the free dofs, and the assembled **matrix**/**vector** with its
**`Inverse`** — are the vocabulary of every later unit. `Solve(a * gfu == f, …)` is just
the shorthand once they are familiar; the *iterative* route it took here (preconditioner
+ Krylov) gets unpacked in unit 6.

## 3. The same loop, nonlinear — stretching a cross flag

To show off **robustness**, the masters grab a stiff specimen — a **flag**, a landscape
plate with a cross punched out — clamp one end and **pull** the other. Large deformation
means **geometric nonlinearity**: we use a **hyperelastic (Neo-Hookean)** stored energy
$\psi(F)$ of the deformation gradient $F=I+\nabla u$, and let NGSolve find the displacement
that **minimises the total energy**. That is a nonlinear system, solved by **Newton's
method** (`solvers.Newton`), ramping the pulling traction in many small steps so each Newton
solve stays in its basin. The same *geometry → weak form → solve → draw* loop — only the
weak form is now an energy, and the solve iterates.

In [ ]:
Wx, Wy, t, bw = 9.0, 6.0, 0.5, 1.0                   # a landscape flag: width × height × thickness
plate = Box(Pnt(0, 0, 0), Pnt(Wx, Wy, t))
cx, cy = Wx / 2, Wy / 2
vbar = Box(Pnt(cx - bw/2, cy - 2.0, -0.1), Pnt(cx + bw/2, cy + 2.0, t + 0.1))   # vertical cross bar
hbar = Box(Pnt(cx - 3.0, cy - bw/2, -0.1), Pnt(cx + 3.0, cy + bw/2, t + 0.1))   # horizontal cross bar
flag = plate - vbar - hbar                           # a flag with a cross punched out
flag.faces.Min(X).name = "hold"                      # clamped end
flag.faces.Max(X).name = "pull"                      # the masters pull here
fmesh = Mesh(OCCGeometry(flag).GenerateMesh(maxh=0.7)); fmesh.Curve(1)

E, nu = 200.0, 0.35                                  # Young's modulus, Poisson ratio
mu, lam = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
V = VectorH1(fmesh, order=1, dirichlet="hold")
ud = V.TrialFunction()
F = Id(3) + Grad(ud); J = Det(F); C = F.trans * F
psi = 0.5 * mu * (Trace(C) - 3) - mu * log(J) + 0.5 * lam * log(J)**2   # Neo-Hooke

traction = Parameter(0.0)
elastic = BilinearForm(V, symmetric=True)
elastic += Variation(psi * dx)                       # stored elastic energy
elastic += Variation(-traction * ud[0] * ds("pull"))  # work of the pulling traction

gfd = GridFunction(V); gfd.vec[:] = 0
morph = GridFunction(V); morph.vec[:] = 0            # frame 0: the original, undeformed flag
nsteps = 12                                          # many small load steps → a slow, smooth morph
for k in range(1, nsteps + 1):                       # ramp the load, keeping continuation
    traction.Set(22.0 * k / nsteps)
    solvers.Newton(elastic, gfd, inverse="sparsecholesky", dampfactor=0.5, printing=False)
    morph.AddMultiDimComponent(gfd.vec)              # one morph frame per load step
print(f"the flag stretched by {max(abs(gfd.vec.FV().NumPy())):.2f} units — and held")

We draw it as a **morph**: a multidim field whose frames are the flag at **successive load
steps**, from undeformed to fully pulled. In the webgui, press **play** (or drag the
**multidim** slider) to watch it stretch — the many frames make the animation slow and
smooth, the surface warps with the displacement and is coloured by its magnitude.

In [ ]:
Draw(morph, fmesh, "displacement", deformation=True,
     interpolate_multidim=True, animate=True)

:::{dropdown} 📚 Further reading
:class: further-reading

- **The Poisson solve, step by step** — i-tutorial
  [1.1 First NGSolve example](https://docu.ngsolve.org/latest/i-tutorials/unit-1.1-poisson/poisson.html).
- **Hyperelasticity & Newton** — i-tutorial
  [3D Solid Mechanics](https://docu.ngsolve.org/latest/i-tutorials/wta/elasticity3D.html)
  and the Newton loop in
  [3.3 Nonlinear problems](https://docu.ngsolve.org/latest/i-tutorials/unit-3.3-nonlinear/nonlinear.html);
  a fuller derivation in the
  [SciCADE course — 3D elasticity](https://jschoeberl.github.io/SciCADE-course/unit2-elasticity/elasticity3D.html).
:::

:::{dropdown} 🧠 Quiz — what were the two "solves" really doing?
:class: quiz

Both followed the identical loop **geometry → weak form → solve → draw**. The
**difference** is the weak form: Poisson is **linear** (one matrix solve — here an
iterative `CG`), elasticity is **nonlinear** (an energy whose minimiser we chase with
**Newton**, each step itself a linear solve). Everything else you saw — `H1` vs
`VectorH1`, `dirichlet` boundary tags, a `Preconditioner`, `Variation`, load ramping —
are the building blocks the rest of Part I unpacks one at a time.
:::

**Next:** the masters withdraw. To approach the Beast yourself you first conjure it some
**food** — and learn elementary **geometry & meshing** along the way (unit 2).

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "02-geometry", "2 · Conjuring geometry & taming the mesh 🍫"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))